# Estudio de Mercado con Mapas de Densidad (Registros INEGI)

Este cuaderno de Colab fue desarrollado previamente utilizando inteligencia artificial y registros del INEGI (Instituto Nacional de Estadística y Geografía de México) para realizar un estudio de mercado. Su objetivo principal es generar mapas de densidad que permitan identificar visualmente la concentración de negocios. Esta herramienta es crucial para buscar clientes potenciales para un negocio de paneles solares, ya que ayuda a localizar áreas con alta actividad comercial donde la demanda de soluciones energéticas podría ser elevada.

Se restringió el area a Zapopan por facilidad de procesamiento, el tamaño de los datos de toda la ciudad de GDL era muy grande.

In [4]:
# --- 1. CONFIGURACIÓN ---
# Lista de configuraciones de mapas. Cada diccionario define un mapa con sus archivos de entrada y nombre de salida.
map_configurations = [
    {
        'name': '6a50p',
        'file_paths': [
            '/content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_6a10p.csv',
            '/content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_11a30p.csv',
            '/content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_31a50p.csv'
        ],
        'output_name': 'mapa_combinado_6a50p.html'
    },
    {
        'name': '6a30p',
        'file_paths': [
            '/content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_6a10p.csv',
            '/content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_11a30p.csv'
        ],
        'output_name': 'mapa_combinado_6a30p.html'
    },
    {
        'name': '6a10p',
        'file_paths': [
            '/content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_6a10p.csv'
        ],
        'output_name': 'mapa_combinado_6a10p.html'
    }
]
# --- 2. EL RESTO DEL CÓDIGO ---
# Importar bibliotecas
import pandas as pd
import folium
from folium.plugins import HeatMap
from google.colab import drive
import sys # Para usar exit()
import os

print("--- Iniciando Proceso para Generación de Mapas Combinados ---")

# Montar Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Error al montar Drive: {e}")

# SCIAN prefixes for filtering, common to all maps
scian_prefixes = ['31', '32', '33', '43', '46', '72', '62']
scian_col_name = 'Código de la clase de actividad SCIAN'

for config in map_configurations:
    map_name = config['name']
    current_paths_archivos = config['file_paths']
    output_html_name = config['output_name']

    print(f"\n--- Procesando Mapa: {map_name} (Archivo de salida: {output_html_name}) ---")

    # --- Fase 1: Cargar y Combinar Datos ---
    lista_de_dataframes = []

    print("Cargando archivos...")
    for file_path in current_paths_archivos:
        if not os.path.exists(file_path):
            print(f"¡ADVERTENCIA! No se encontró el archivo: {file_path}")
            print("Este archivo será omitido. Verifica la ruta y el nombre.")
            continue

        try:
            df = pd.read_csv(file_path, encoding='latin1')
            lista_de_dataframes.append(df)
            print(f"Cargado: {file_path} ({len(df)} filas)")
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(file_path, encoding='ISO-8859-1')
                lista_de_dataframes.append(df)
                print(f"Cargado: {file_path} ({len(df)} filas)")
            except Exception as e:
                print(f"Error al leer el archivo {file_path}: {e}")
        except Exception as e:
            print(f"Error al leer el archivo {file_path}: {e}")

    if not lista_de_dataframes:
        print(f"¡ADVERTENCIA! No se pudo cargar ningún archivo para el mapa {map_name}. Se omitirá este mapa.")
        continue # Skip to the next map configuration

    # Combinar todos los dataframes en uno solo
    df_combined = pd.concat(lista_de_dataframes)
    print(f"Total de negocios combinados para {map_name}: {len(df_combined)}")

    # --- Fase 2: Pulir (Filtrar por Industria SCIAN) ---
    if scian_col_name not in df_combined.columns:
        print(f"Error: La columna '{scian_col_name}' no se encontró en los datos combinados para el mapa {map_name}.")
        print("Se omitirá este mapa.")
        continue

    df_combined[scian_col_name] = df_combined[scian_col_name].astype(str)
    mask = df_combined[scian_col_name].str.startswith(tuple(scian_prefixes), na=False)
    df_filtrado = df_combined[mask]

    print(f"Fase 2 (Pulido) completa para {map_name}. {len(df_filtrado)} negocios de interés encontrados.")

    # --- Fase 3: Modelar (Generar el Mapa Interactivo) ---
    df_mapa = df_filtrado.dropna(subset=['Latitud', 'Longitud'])

    if not df_mapa.empty:
        map_center = [df_mapa['Latitud'].mean(), df_mapa['Longitud'].mean()]

        m = folium.Map(location=map_center, zoom_start=12, tiles="CartoDB dark_matter")

        heat_data = [[row['Latitud'], row['Longitud']] for index, row in df_mapa.iterrows()]

        # Ajustar el radio y blur puede ser útil si tienes muchos más puntos
        HeatMap(heat_data, radius=12, blur=18).add_to(m)

        # Guardar el mapa
        m.save(output_html_name)

        print(f"¡ÉXITO! Se ha generado el mapa '{map_name}'.")
        print(f"El archivo se llama: {output_html_name}")
        print(f"Lo encontrarás en la carpeta de archivos de Colab (panel izquierdo). ¡Descárgalo!")

    else:
        print(f"No se encontraron negocios con coordenadas válidas para el mapa {map_name} y este filtro.")

print("\n--- Proceso de Generación de Mapas Completado ---")

--- Iniciando Proceso para Generación de Mapas Combinados ---
Mounted at /content/drive

--- Procesando Mapa: 6a50p (Archivo de salida: mapa_combinado_6a50p.html) ---
Cargando archivos...
Cargado: /content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_6a10p.csv (3841 filas)
Cargado: /content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_11a30p.csv (2745 filas)
Cargado: /content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_31a50p.csv (570 filas)
Total de negocios combinados para 6a50p: 7156
Fase 2 (Pulido) completa para 6a50p. 5972 negocios de interés encontrados.
¡ÉXITO! Se ha generado el mapa '6a50p'.
El archivo se llama: mapa_combinado_6a50p.html
Lo encontrarás en la carpeta de archivos de Colab (panel izquierdo). ¡Descárgalo!

--- Procesando Mapa: 6a30p (Archivo de salida: mapa_combinado_6a30p.html) ---
Cargando archivos...
Cargado: /content/drive/MyDrive/Colab Notebooks/INEGI_DENUE_26102025_6a10p.csv (3841 filas)
Cargado: /content/drive/MyDrive/Colab Notebooks/INEGI